# Role-Skill Frequency Analytical Mart

**Purpose**: Calculates the frequency of skill mentions and demand scores per canonical role.

**Target Table**: `workspace.reporting.reporting_role_skill_frequency`

**Data Sources**:
- `workspace.gold.bridge_job_skill`
- `workspace.gold.fact_job_postings`
- `workspace.gold.dim_role`
- `workspace.gold.dim_skill`

In [0]:
# Configuration
CATALOG = "workspace"
TARGET_TABLE = f"{CATALOG}.reporting.reporting_role_skill_frequency"
METADATA_TABLE = f"{CATALOG}.metadata.reporting_role_skill_frequency_refresh_log"

In [0]:
import uuid
from datetime import datetime

run_id = str(uuid.uuid4())
run_timestamp = datetime.now()

# Ensure refresh audit log exists
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {METADATA_TABLE} (
  run_id STRING NOT NULL,
  run_timestamp TIMESTAMP NOT NULL,
  status STRING NOT NULL,
  rows_processed BIGINT,
  processing_time_seconds DECIMAL(10,2),
  error_message STRING
)
USING DELTA
COMMENT 'Audit log for reporting_role_skill_frequency refreshes'
""")

In [0]:
import time

start_time = time.time()
print(f"Refreshing role-skill frequency mart: {TARGET_TABLE}...")

try:
    refresh_sql = f"""
    INSERT OVERWRITE TABLE {TARGET_TABLE}
    WITH role_postings AS (
      SELECT
        role_sk,
        COUNT(DISTINCT fact_job_posting_sk) AS total_role_postings
      FROM {CATALOG}.gold.fact_job_postings
      WHERE active_flag = TRUE AND role_sk IS NOT NULL AND role_sk != -1
      GROUP BY role_sk
    ),
    role_skill_counts AS (
      SELECT
        f.role_sk,
        bjs.skill_sk,
        COUNT(DISTINCT f.fact_job_posting_sk) AS postings_count,
        AVG(bjs.extraction_confidence) AS avg_extraction_confidence,
        AVG(
          CASE 
            WHEN bjs.skill_importance = 'REQUIRED' THEN 5.0
            WHEN bjs.skill_importance = 'PREFERRED' THEN 3.0
            WHEN bjs.skill_importance = 'NICE_TO_HAVE' THEN 1.0
            ELSE 1.0
          END
        ) AS avg_demand_score
      FROM {CATALOG}.gold.bridge_job_skill bjs
      JOIN {CATALOG}.gold.fact_job_postings f ON bjs.job_sk = f.job_sk
      WHERE f.active_flag = TRUE AND f.role_sk IS NOT NULL AND f.role_sk != -1
      GROUP BY f.role_sk, bjs.skill_sk
    )
    SELECT
      rsc.role_sk,
      r.role_key,
      rsc.skill_sk,
      s.skill_key,
      rsc.postings_count,
      rp.total_role_postings,
      ROUND((rsc.postings_count / rp.total_role_postings) * 100, 2) AS mention_percentage,
      ROUND(rsc.avg_extraction_confidence, 4) AS avg_extraction_confidence,
      ROUND(rsc.avg_demand_score, 2) AS avg_demand_score,
      CURRENT_TIMESTAMP() AS updated_at
    FROM role_skill_counts rsc
    JOIN role_postings rp ON rsc.role_sk = rp.role_sk
    JOIN {CATALOG}.gold.dim_role r ON rsc.role_sk = r.role_sk
    JOIN {CATALOG}.gold.dim_skill s ON rsc.skill_sk = s.skill_sk
    """
    
    # Execute query
    spark.sql(refresh_sql)
    duration = time.time() - start_time
    
    # Log success
    spark.sql(f"""
    INSERT INTO {METADATA_TABLE} VALUES
    ('{run_id}', '{run_timestamp}', 'SUCCESS', NULL, {duration}, NULL)
    """)
    print(f"✓ Refresh complete in {duration:.2f}s")
    
except Exception as e:
    duration = time.time() - start_time
    err = str(e).replace("'", "''")
    # Log failure
    spark.sql(f"""
    INSERT INTO {METADATA_TABLE} VALUES
    ('{run_id}', '{run_timestamp}', 'FAILED', NULL, {duration}, '{err}')
    """)
    raise e